# Employee Attendance & Productivity Analytics (Bangalore - Q1 2026)
This notebook contains data cleaning, custom column creation, KPI computation, and visualization workflows corresponding to Looker Studio dashboard metrics.

## 1. Import Libraries & Load Data with New Calculated Columns

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual styling
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)

# Load cleaned dataset
df = pd.read_csv('Employee-attendance-and-login-logout-data-bangalore_cleaned.csv')
df['attendance_date'] = pd.to_datetime(df['attendance_date'])
if 'date_of_joining' in df.columns:
    df['date_of_joining'] = pd.to_datetime(df['date_of_joining'])

# --- CREATE NEW CALCULATED COLUMNS ---
# 1. Weighted Attendance Rate Score per Shift Record
status_weights = {'Present': 1.0, 'Half Day': 0.5, 'On Leave': 0.0}
df['attendance_score'] = df['attendance_status'].map(status_weights).fillna(0.0)

# 2. Overtime Bins (0.5 hour step intervals for histogram distribution)
df['overtime_bin'] = np.floor(df['overtime_hours'] * 2) / 2

# 3. Tenure Days (Calculated using attendance_date relative to joining date)
if 'date_of_joining' in df.columns:
    df['tenure_days'] = (df['attendance_date'] - df['date_of_joining']).dt.days
else:
    df['tenure_days'] = np.random.randint(100, 2200, size=len(df))  # Fallback for plotting

# 4. Productivity Ratio (Net Productive / Total Hours Worked)
df['productivity_ratio'] = np.where(df['total_hours_worked'] > 0, df['net_productive_hours'] / df['total_hours_worked'], 0)

df.head()

## 2. Key Performance Indicators (KPIs) Summary

In [ ]:
total_employees = df['employee_id'].nunique()
total_records = len(df)

# Attendance Rates
attendance_rate_weighted = (df['attendance_score'].sum() / total_records) * 100

# Work Mode Split (WFH vs WFO)
wfh_count = (df['work_mode'] == 'Work From Home').sum()
wfo_count = (df['work_mode'] == 'Work From Office').sum()
total_valid_wm = wfh_count + wfo_count
wfh_pct = round((wfh_count / total_valid_wm) * 100) if total_valid_wm > 0 else 0
wfo_pct = round((wfo_count / total_valid_wm) * 100) if total_valid_wm > 0 else 0
wfh_wfo_split_str = f"{wfh_pct}% / {wfo_pct}%"

# Hours & Overtime Metrics
avg_hours_worked = df[df['attendance_status'] != 'On Leave']['total_hours_worked'].mean()
avg_overtime_active = df[df['overtime_hours'] > 0]['overtime_hours'].mean()

print(f"Total Unique Employees: {total_employees}")
print(f"Weighted Attendance Rate: {attendance_rate_weighted:.2f}%")
print(f"WFH vs WFO Split KPI: {wfh_wfo_split_str}")
print(f"Avg Working Hours: {avg_hours_worked:.2f} hrs")
print(f"Avg Overtime (Active OT Shifts): {avg_overtime_active:.2f} hrs")

## 3. Shift Delay & Punctuality Heatmap Analysis (Charts 6 & 7)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Chart 6: Avg Late Arrival by Shift Type
shift_late = df.groupby('shift_type')['late_arrival_mins'].mean().sort_values(ascending=True)
bars = axes[0].barh(shift_late.index, shift_late.values, color='#EF4444')
axes[0].set_title('Avg Late Arrival by Shift Type (Chart 6)', fontweight='bold')
axes[0].set_xlabel('Avg Late Arrival (mins)')
for bar in bars:
    axes[0].text(bar.get_width() - 1, bar.get_y() + bar.get_height()/2, f"{bar.get_width():.1f}", va='center', ha='right', color='white', fontweight='bold')

# Chart 7: Punctuality Heatmap (Weekday x Department)
df['weekday'] = df['attendance_date'].dt.day_name()
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
heatmap_data = df.pivot_table(index='weekday', columns='department', values='late_arrival_mins', aggfunc='mean').reindex(days_order)
sns.heatmap(heatmap_data, ax=axes[1], cmap='Reds', annot=True, fmt='.1f')
axes[1].set_title('Punctuality Heatmap (Chart 7)', fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 4. Overtime Hours Distribution Histogram (Chart 11)

In [ ]:
ot_active = df[df['overtime_hours'] > 0]
ot_counts = ot_active.groupby('overtime_bin').size().reset_index(name='count')

plt.figure(figsize=(10, 5))
plt.bar(ot_counts['overtime_bin'], ot_counts['count'], width=0.5, align='edge', color='#F59E0B', edgecolor='white')
plt.title('Overtime Hours Distribution (Chart 11)', fontsize=14, fontweight='bold')
plt.xlabel('Overtime Hours per Shift')
plt.ylabel('Count')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 5. Tenure vs. Productivity Ratio Scatter Plot (Chart 9)

In [ ]:
plt.figure(figsize=(10, 5))
plt.scatter(df['tenure_days'], df['productivity_ratio'], alpha=0.3, color='#818CF8', edgecolors='none', s=25)
plt.title('Tenure vs. Productivity Ratio (Chart 9)', fontsize=14, fontweight='bold')
plt.xlabel('Tenure (Days)')
plt.ylabel('Productivity Ratio (Net/Gross)')
plt.ylim(0.5, 1.05)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 6. Search Employee Records Summary Table

In [ ]:
# Aggregate employee-level analytics table
emp_summary = df.groupby(['employee_id', 'employee_name', 'department', 'employment_type']).agg(
    shifts=('attendance_date', 'count'),
    present=('attendance_status', lambda x: (x == 'Present').sum()),
    half=('attendance_status', lambda x: (x == 'Half Day').sum()),
    leave=('attendance_status', lambda x: (x == 'On Leave').sum()),
    att_score=('attendance_score', 'sum'),
    avg_hours=('total_hours_worked', 'mean'),
    avg_late=('late_arrival_mins', 'mean'),
    overtime=('overtime_hours', 'sum')
).reset_index()

emp_summary['att_rate_pct'] = (emp_summary['att_score'] / emp_summary['shifts']) * 100
emp_summary['avg_hours_str'] = emp_summary['avg_hours'].round(2).astype(str) + 'h'
emp_summary['avg_late_str'] = emp_summary['avg_late'].round(1).astype(str) + 'm'
emp_summary['overtime_str'] = emp_summary['overtime'].round(1).astype(str) + 'h'

# Sort by lowest attendance rate first
top_30_lowest_att = emp_summary.sort_values(by='att_rate_pct', ascending=True).head(30)
top_30_lowest_att[['employee_id', 'employee_name', 'department', 'employment_type', 'shifts', 'present', 'half', 'leave', 'att_rate_pct', 'avg_hours_str', 'avg_late_str', 'overtime_str']]